# Free-Tier Cloud Model Adaptation Pipeline (Kaggle / Google Colab)

This notebook executes the production 4-bit QLoRA fine-tuning cycle for **Qwen/Qwen2.5-7B-Instruct** using curated synthetic self-correction evolution pairs and research papers.

### Hardware Requirements:
- Free NVIDIA T4 GPU (16GB VRAM) or P100 GPU on Google Colab or Kaggle.
- Enforces memory footprint strictly **< 14GB VRAM** using:
  - 4-bit NormalFloat4 (NF4) double quantization via `bitsandbytes`
  - Gradient checkpointing (`gradient_checkpointing=True`)
  - Cache disabled during training (`model.config.use_cache = False`)
  - `per_device_train_batch_size = 1` with `gradient_accumulation_steps = 4`
  - LoRA adapters on attention projection matrices (`q_proj`, `k_proj`, `v_proj`, `o_proj`)

In [ ]:
# Step 1: Install Dependencies
!pip install -q --upgrade pip
!pip install -q torch transformers peft bitsandbytes trl accelerate datasets google-cloud-storage sentencepiece

In [ ]:
# Step 2: Verify GPU Acceleration and CUDA Memory
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: No GPU detected. In Colab, go to Runtime -> Change runtime type -> T4 GPU.")

In [ ]:
# Step 3: Run Full QLoRA Fine-Tuning Pipeline
import os
import sys
from pathlib import Path

# If running inside cloned repo, add to path; otherwise download script
if not os.path.exists("training/kaggle_lora_train.py"):
    !git clone https://github.com/Rajuzet/AGI-pack.git
    %cd AGI-pack

# Execute the adaptation pipeline with 4-bit NF4 quantization
!python training/kaggle_lora_train.py \
    --model-id "Qwen/Qwen2.5-7B-Instruct" \
    --epochs 1 \
    --batch-size 1 \
    --grad-accum 4 \
    --lora-r 16 \
    --lora-alpha 32 \
    --output-dir "./model_checkpoints" \
    --data-dir "./data_staging"

In [ ]:
# Step 4: Stream Adapter Weights to Google Cloud Storage
from google.cloud import storage
from google.colab import auth

# Authenticate GCP user account if running in Google Colab
try:
    auth.authenticate_user()
    print("Authenticated GCP User successfully in Colab.")
except Exception:
    print("Standard GCS credentials or Kaggle secret will be used.")

BUCKET_NAME = os.environ.get("GCS_BUCKET_NAME", "agi-agent-ingestion-data")
TARGET_PREFIX = "models/lora_checkpoints/latest"

def upload_adapter_to_gcs(local_dir, bucket_name, prefix):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    for root, _, files in os.walk(local_dir):
        for f in files:
            local_path = os.path.join(root, f)
            rel = os.path.relpath(local_path, local_dir)
            blob = bucket.blob(f"{prefix}/{rel}")
            blob.upload_from_filename(local_path)
            print(f"Uploaded {rel} -> gs://{bucket_name}/{prefix}/{rel}")

# Find latest checkpoint directory
checkpoints = sorted(Path("./model_checkpoints").glob("run_*"))
if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f"Streaming latest checkpoint {latest_checkpoint} to GCS...")
    try:
        upload_adapter_to_gcs(latest_checkpoint, BUCKET_NAME, TARGET_PREFIX)
        print(f"SUCCESS: Adapter deployed to gs://{BUCKET_NAME}/{TARGET_PREFIX}/")
    except Exception as exc:
        print(f"GCS upload notice: {exc}")
else:
    print("No checkpoint found in ./model_checkpoints.")